# World Cup Score Predictor — Clean Feature Engineering Notebook

This notebook builds a clean pre-match feature dataset for the World Cup score predictor.

Main goals:
- One row per match.
- Use only information available before kickoff.
- Convert post-match ELO/rank values into pre-match values.
- Keep a focused, relevant feature set.
- Evaluate with expanding-window World Cup backtesting.

## 1. Imports and Paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import exp, factorial

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 120)

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUTS_DIR = BASE_DIR / "outputs" / "evaluation"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)

RAW_DIR: ../data/raw
PROCESSED_DIR: ../data/processed
OUTPUTS_DIR: ../outputs/evaluation


## 2. Load Yearly ELO Match Files

In [2]:
elo_files = sorted(RAW_DIR.glob("elo_*_results.csv"))

if not elo_files:
    raise FileNotFoundError("No files found matching data/raw/elo_*_results.csv")

dfs = []
for file in elo_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    dfs.append(temp)

elo_df = pd.concat(dfs, ignore_index=True)
elo_df["date"] = pd.to_datetime(elo_df["date"], errors="coerce")

before_filter_shape = elo_df.shape

elo_df = elo_df[
    elo_df["date"].dt.year >= 2004
].copy()

elo_df = elo_df.sort_values("date").reset_index(drop=True)

print("Filtered ELO data to 2004+")
print("Before:", before_filter_shape)
print("After:", elo_df.shape)
print("Date range:", elo_df["date"].min().date(), "to", elo_df["date"].max().date())

print("Number of ELO files:", len(elo_files))
print("Loaded ELO shape:", elo_df.shape)
display(elo_df.head())

Filtered ELO data to 2004+
Before: (24261, 16)
After: (21540, 16)
Date range: 2004-01-01 to 2026-06-15
Number of ELO files: 26
Loaded ELO shape: (21540, 16)


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,rating_b,rank_change_a,rank_change_b,rank_a,rank_b,source_file
0,2004-01-01,Bermuda,Barbados,0,4,Dudley Eve Memorial Trophy,Bermuda,-28,28,1242,1397,-8,7,146,109,elo_2004_results.csv
1,2004-01-01,Kuwait,Yemen,4,0,Gulf Cup,Kuwait,6,-6,1525,1173,2,-1,72,160,elo_2004_results.csv
2,2004-01-01,Saudi Arabia,Bahrain,1,0,Gulf Cup,Kuwait,12,-12,1674,1511,3,-6,44,78,elo_2004_results.csv
3,2004-01-03,Bahrain,Oman,1,0,Gulf Cup,Kuwait,23,-23,1534,1537,7,-4,71,70,elo_2004_results.csv
4,2004-01-03,Qatar,United Arab Emirates,0,0,Gulf Cup,Kuwait,-4,4,1517,1459,-4,1,77,90,elo_2004_results.csv


## 3. Basic Cleaning and No-Leakage Pre-Match Features

In [3]:
df = elo_df.copy()

required_cols = [
    "date", "team_a", "team_b", "goals_a", "goals_b", "competition", "location",
    "rating_change_a", "rating_change_b", "rating_a", "rating_b",
    "rank_change_a", "rank_change_b", "rank_a", "rank_b"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "team_a", "team_b", "goals_a", "goals_b"])
df = df.sort_values("date").reset_index(drop=True)

# Targets
df["target_goals_a"] = df["goals_a"].astype(int)
df["target_goals_b"] = df["goals_b"].astype(int)
df["target_goal_diff"] = df["target_goals_a"] - df["target_goals_b"]
df["target_total_goals"] = df["target_goals_a"] + df["target_goals_b"]

# Result points - needed for rolling form features
df["result_a"] = np.where(
    df["goals_a"] > df["goals_b"],
    3,
    np.where(df["goals_a"] == df["goals_b"], 1, 0)
)

df["result_b"] = np.where(
    df["goals_b"] > df["goals_a"],
    3,
    np.where(df["goals_a"] == df["goals_b"], 1, 0)
)

# Raw rating/rank values are post-match, so reconstruct pre-match values
df["rating_a_before"] = df["rating_a"] - df["rating_change_a"]
df["rating_b_before"] = df["rating_b"] - df["rating_change_b"]

df["rank_a_before"] = df["rank_a"] - df["rank_change_a"]
df["rank_b_before"] = df["rank_b"] - df["rank_change_b"]

df["elo_diff"] = df["rating_a_before"] - df["rating_b_before"]
df["rank_diff"] = df["rank_a_before"] - df["rank_b_before"]

print("Cleaned shape:", df.shape)

display(df[
    [
        "date",
        "team_a",
        "team_b",
        "competition",
        "location",
        "rating_a_before",
        "rating_b_before",
        "elo_diff",
        "rank_a_before",
        "rank_b_before",
        "rank_diff",
        "target_goals_a",
        "target_goals_b",
    ]
].head())

Cleaned shape: (21540, 28)


,date,team_a,team_b,competition,location,rating_a_before,rating_b_before,elo_diff,rank_a_before,rank_b_before,rank_diff,target_goals_a,target_goals_b
0,2004-01-01,Bermuda,Barbados,Dudley Eve Memorial Trophy,Bermuda,1270,1369,-99,154,102,52,0,4
1,2004-01-01,Kuwait,Yemen,Gulf Cup,Kuwait,1519,1179,340,70,161,-91,4,0
2,2004-01-01,Saudi Arabia,Bahrain,Gulf Cup,Kuwait,1662,1523,139,41,84,-43,1,0
3,2004-01-03,Bahrain,Oman,Gulf Cup,Kuwait,1511,1560,-49,64,74,-10,1,0
4,2004-01-03,Qatar,United Arab Emirates,Gulf Cup,Kuwait,1521,1455,66,81,89,-8,0,0


## 4. Rolling Team Features

In [4]:
WINDOW = 5
team_history = {}

ELO_BASE = df[["rating_a_before", "rating_b_before"]].stack().mean()

features = {
    "team_a_weighted_goals_for_avg_last5": [],
    "team_b_weighted_goals_for_avg_last5": [],
    "team_a_weighted_goals_against_avg_last5": [],
    "team_b_weighted_goals_against_avg_last5": [],
    "team_a_avg_opponent_elo_last5": [],
    "team_b_avg_opponent_elo_last5": [],
    "team_a_rating_change_avg_last5": [],
    "team_b_rating_change_avg_last5": [],
    "team_a_matches_played_before": [],
    "team_b_matches_played_before": [],
    "team_a_days_since_last_match": [],
    "team_b_days_since_last_match": [],
}

def mean_or_zero(values):
    return float(np.mean(values)) if len(values) > 0 else 0.0

def mean_or_base(values, base_value):
    values = list(values)
    return float(np.mean(values)) if values else float(base_value)

for _, row in df.iterrows():
    team_a = row["team_a"]
    team_b = row["team_b"]
    date = row["date"]

    team_history.setdefault(team_a, [])
    team_history.setdefault(team_b, [])

    hist_a_all = team_history[team_a]
    hist_b_all = team_history[team_b]

    hist_a = hist_a_all[-WINDOW:]
    hist_b = hist_b_all[-WINDOW:]

    features["team_a_weighted_goals_for_avg_last5"].append(
        mean_or_zero([x["weighted_goals_for"] for x in hist_a])
    )

    features["team_b_weighted_goals_for_avg_last5"].append(
        mean_or_zero([x["weighted_goals_for"] for x in hist_b])
    )

    features["team_a_weighted_goals_against_avg_last5"].append(
        mean_or_zero([x["weighted_goals_against"] for x in hist_a])
    )

    features["team_b_weighted_goals_against_avg_last5"].append(
        mean_or_zero([x["weighted_goals_against"] for x in hist_b])
    )

    features["team_a_avg_opponent_elo_last5"].append(
        mean_or_base(
            [x["opponent_elo"] for x in hist_a],
            ELO_BASE,
        )
    )

    features["team_b_avg_opponent_elo_last5"].append(
        mean_or_base(
            [x["opponent_elo"] for x in hist_b],
            ELO_BASE,
        )
    )

    features["team_a_rating_change_avg_last5"].append(
        mean_or_zero([x["rating_change"] for x in hist_a])
    )

    features["team_b_rating_change_avg_last5"].append(
        mean_or_zero([x["rating_change"] for x in hist_b])
    )

    features["team_a_matches_played_before"].append(len(hist_a_all))
    features["team_b_matches_played_before"].append(len(hist_b_all))

    features["team_a_days_since_last_match"].append(
        (date - hist_a_all[-1]["date"]).days if hist_a_all else 999
    )

    features["team_b_days_since_last_match"].append(
        (date - hist_b_all[-1]["date"]).days if hist_b_all else 999
    )

    opponent_weight_for_a = row["rating_b_before"] / ELO_BASE
    opponent_weight_for_b = row["rating_a_before"] / ELO_BASE

    team_history[team_a].append({
        "date": date,
        "weighted_goals_for": row["goals_a"] * opponent_weight_for_a,
        "weighted_goals_against": row["goals_b"] * (ELO_BASE / row["rating_b_before"]),
        "opponent_elo": row["rating_b_before"],
        "rating_change": row["rating_change_a"],
    })

    team_history[team_b].append({
        "date": date,
        "weighted_goals_for": row["goals_b"] * opponent_weight_for_b,
        "weighted_goals_against": row["goals_a"] * (ELO_BASE / row["rating_a_before"]),
        "opponent_elo": row["rating_a_before"],
        "rating_change": row["rating_change_b"],
    })

for col, values in features.items():
    df[col] = values

print("Rolling features created.")
print("ELO_BASE:", round(ELO_BASE, 2))

Rolling features created.
ELO_BASE: 1480.69


## 5. Tournament State Features

In [5]:
df["tournament_year"] = df["date"].dt.year
df["tournament_key"] = df["competition"].astype(str) + "_" + df["tournament_year"].astype(str)

tournament_states = {}

tournament_features = {
    "team_a_tournament_matches_played": [],
    "team_b_tournament_matches_played": [],
    "tournament_points_diff": [],
    "tournament_goal_diff_diff": [],
}

def empty_state():
    return {
        "matches": 0,
        "points": 0,
        "goals_for": 0,
        "goals_against": 0,
        "goal_diff": 0,
    }

for _, row in df.iterrows():
    key = row["tournament_key"]
    team_a = row["team_a"]
    team_b = row["team_b"]

    tournament_states.setdefault(key, {})
    tournament_states[key].setdefault(team_a, empty_state())
    tournament_states[key].setdefault(team_b, empty_state())

    state_a = tournament_states[key][team_a]
    state_b = tournament_states[key][team_b]

    tournament_features["team_a_tournament_matches_played"].append(
        state_a["matches"]
    )

    tournament_features["team_b_tournament_matches_played"].append(
        state_b["matches"]
    )

    tournament_features["tournament_points_diff"].append(
        state_a["points"] - state_b["points"]
    )

    tournament_features["tournament_goal_diff_diff"].append(
        state_a["goal_diff"] - state_b["goal_diff"]
    )

    state_a["matches"] += 1
    state_a["points"] += row["result_a"]
    state_a["goals_for"] += row["goals_a"]
    state_a["goals_against"] += row["goals_b"]
    state_a["goal_diff"] = (
        state_a["goals_for"] - state_a["goals_against"]
    )

    state_b["matches"] += 1
    state_b["points"] += row["result_b"]
    state_b["goals_for"] += row["goals_b"]
    state_b["goals_against"] += row["goals_a"]
    state_b["goal_diff"] = (
        state_b["goals_for"] - state_b["goals_against"]
    )

for col, values in tournament_features.items():
    df[col] = values

print("Tournament state features created.")
display(df[[
    "date",
    "competition",
    "team_a",
    "team_b",
    "team_a_tournament_matches_played",
    "team_b_tournament_matches_played",
    "tournament_points_diff",
    "tournament_goal_diff_diff",
]].head(12))

Tournament state features created.


,date,competition,team_a,team_b,team_a_tournament_matches_played,team_b_tournament_matches_played,tournament_points_diff,tournament_goal_diff_diff
0,2004-01-01,Dudley Eve Memorial Trophy,Bermuda,Barbados,0,0,0,0
1,2004-01-01,Gulf Cup,Kuwait,Yemen,0,0,0,0
2,2004-01-01,Gulf Cup,Saudi Arabia,Bahrain,0,0,0,0
3,2004-01-03,Gulf Cup,Bahrain,Oman,1,0,0,-1
4,2004-01-03,Gulf Cup,Qatar,United Arab Emirates,0,0,0,0
5,2004-01-04,Gulf Cup,Kuwait,Saudi Arabia,1,1,0,3
6,2004-01-05,Gulf Cup,Qatar,Yemen,1,1,1,4
7,2004-01-06,Gulf Cup,Saudi Arabia,Oman,2,1,4,2
8,2004-01-07,Gulf Cup,Bahrain,United Arab Emirates,2,1,2,0
9,2004-01-08,Friendly,Egypt,Rwanda,0,0,0,0


## 6. Competition Importance + Transfermarkt Market Values

In [6]:
# =========================
# Transfermarkt Market Values
# =========================

tm_path = PROCESSED_DIR / "transfermarkt_market_values_clean.csv"

if not tm_path.exists():
    raise FileNotFoundError(
        f"Missing Transfermarkt file: {tm_path}. "
        "Run 02_feature_engineering.ipynb first."
    )

tm = pd.read_csv(tm_path)

required_tm_cols = [
    "team_name_tm",
    "season_id",
    "avg_player_value_millions_eur",
    "market_value_relative_to_year_mean",
    "log_market_value",
]

missing_tm = [col for col in required_tm_cols if col not in tm.columns]

if missing_tm:
    raise ValueError(
        f"Missing Transfermarkt columns: {missing_tm}"
    )

tm = tm[required_tm_cols].copy()

name_map = {
    "USA": "United States",
    "Turkey": "Turkiye",
    "Czech Republic": "Czechia",
    "Cote d'Ivoire": "Ivory Coast",
    "Democratic Republic of the Congo": "DR Congo",
    "Korea Republic": "South Korea",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Cape Verde Islands": "Cape Verde",
    "Curacao": "Curaçao",
    "Macedonia": "North Macedonia",
    "Swaziland": "Eswatini",
    "Sao Tome and Principe": "São Tomé and Príncipe",
}

teams_tm = set(tm["team_name_tm"].unique())

valid_name_map = {
    src: dst
    for src, dst in name_map.items()
    if dst in teams_tm
}

invalid_name_map = {
    src: dst
    for src, dst in name_map.items()
    if dst not in teams_tm
}

print("Valid Transfermarkt name mappings:")
print(valid_name_map)

print("Invalid / not found Transfermarkt name mappings:")
print(invalid_name_map)

df["season_id"] = df["date"].dt.year

df["team_a_tm"] = df["team_a"].replace(valid_name_map)
df["team_b_tm"] = df["team_b"].replace(valid_name_map)

tm_a = tm.rename(columns={
    "team_name_tm": "team_a_tm",
    "avg_player_value_millions_eur": "avg_player_value_a",
    "market_value_relative_to_year_mean": "market_value_rel_mean_a",
    "log_market_value": "log_market_value_a",
})

tm_b = tm.rename(columns={
    "team_name_tm": "team_b_tm",
    "avg_player_value_millions_eur": "avg_player_value_b",
    "market_value_relative_to_year_mean": "market_value_rel_mean_b",
    "log_market_value": "log_market_value_b",
})

df = df.merge(
    tm_a,
    on=["team_a_tm", "season_id"],
    how="left",
)

df = df.merge(
    tm_b,
    on=["team_b_tm", "season_id"],
    how="left",
)

print("Market value coverage before fill:")
print(
    "team_a missing:",
    round(df["avg_player_value_a"].isna().mean(), 3)
)
print(
    "team_b missing:",
    round(df["avg_player_value_b"].isna().mean(), 3)
)

market_cols = [
    "avg_player_value_a",
    "avg_player_value_b",
    "market_value_rel_mean_a",
    "market_value_rel_mean_b",
    "log_market_value_a",
    "log_market_value_b",
]

for col in market_cols:
    df[col] = (
        df[col]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

df["avg_player_value_diff"] = (
    df["avg_player_value_a"]
    - df["avg_player_value_b"]
)

df["market_value_rel_mean_diff"] = (
    df["market_value_rel_mean_a"]
    - df["market_value_rel_mean_b"]
)

print("Market value features created.")

Valid Transfermarkt name mappings:
{'USA': 'United States', 'Turkey': 'Turkiye', 'Czech Republic': 'Czechia', "Cote d'Ivoire": 'Ivory Coast', 'Korea Republic': 'South Korea', 'Cape Verde Islands': 'Cape Verde', 'Curacao': 'Curaçao', 'Macedonia': 'North Macedonia', 'Swaziland': 'Eswatini', 'Sao Tome and Principe': 'São Tomé and Príncipe'}
Invalid / not found Transfermarkt name mappings:
{'Democratic Republic of the Congo': 'DR Congo', 'Bosnia-Herzegovina': 'Bosnia and Herzegovina'}
Market value coverage before fill:
team_a missing: 0.034
team_b missing: 0.04
Market value features created.


## 6.5. Transfermarkt Position-Level Features

In [7]:
position_values_path = PROCESSED_DIR / "transfermarkt_position_values_2004_2026.csv"

if not position_values_path.exists():
    raise FileNotFoundError(
        f"Missing position values file: {position_values_path}"
    )

pv = pd.read_csv(position_values_path)

required_pv_cols = [
    "team_name_tm",
    "season_id",
    "goalkeeper_market_value_millions_eur",
    "defender_market_value_millions_eur",
    "midfield_market_value_millions_eur",
    "attack_market_value_millions_eur",
    "goalkeeper_avg_age",
    "defender_avg_age",
    "midfield_avg_age",
    "attack_avg_age",
    "scraped_total_market_value_millions_eur",
]

missing_pv = [c for c in required_pv_cols if c not in pv.columns]
if missing_pv:
    raise ValueError(f"Missing position value columns: {missing_pv}")

pv = pv[required_pv_cols].copy()

pv_a = pv.rename(columns={
    "team_name_tm": "team_a_tm",
    "goalkeeper_market_value_millions_eur": "goalkeeper_value_a",
    "defender_market_value_millions_eur": "defender_value_a",
    "midfield_market_value_millions_eur": "midfield_value_a",
    "attack_market_value_millions_eur": "attack_value_a",
    "goalkeeper_avg_age": "goalkeeper_age_a",
    "defender_avg_age": "defender_age_a",
    "midfield_avg_age": "midfield_age_a",
    "attack_avg_age": "attack_age_a",
    "scraped_total_market_value_millions_eur": "position_total_value_a",
})

pv_b = pv.rename(columns={
    "team_name_tm": "team_b_tm",
    "goalkeeper_market_value_millions_eur": "goalkeeper_value_b",
    "defender_market_value_millions_eur": "defender_value_b",
    "midfield_market_value_millions_eur": "midfield_value_b",
    "attack_market_value_millions_eur": "attack_value_b",
    "goalkeeper_avg_age": "goalkeeper_age_b",
    "defender_avg_age": "defender_age_b",
    "midfield_avg_age": "midfield_age_b",
    "attack_avg_age": "attack_age_b",
    "scraped_total_market_value_millions_eur": "position_total_value_b",
})

df = df.merge(pv_a, on=["team_a_tm", "season_id"], how="left")
df = df.merge(pv_b, on=["team_b_tm", "season_id"], how="left")

position_cols = [
    "goalkeeper_value_a",
    "goalkeeper_value_b",
    "defender_value_a",
    "defender_value_b",
    "midfield_value_a",
    "midfield_value_b",
    "attack_value_a",
    "attack_value_b",
    "goalkeeper_age_a",
    "goalkeeper_age_b",
    "defender_age_a",
    "defender_age_b",
    "midfield_age_a",
    "midfield_age_b",
    "attack_age_a",
    "attack_age_b",
    "position_total_value_a",
    "position_total_value_b",
]

for col in position_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

def safe_share(value, total):
    return np.where(total > 0, value / total, 0)

def safe_age_diff(age_a, age_b):
    return np.where(
        (age_a > 0) & (age_b > 0),
        age_a - age_b,
        0,
    )
def safe_ratio(numerator, denominator):
    return np.where(
        denominator > 0,
        numerator / denominator,
        0,
    )


def mean_positive_age(gk_age, def_age, mid_age, att_age):
    ages = pd.concat(
        [gk_age, def_age, mid_age, att_age],
        axis=1,
    )

    ages = ages.where(ages > 0)

    return ages.mean(axis=1).fillna(0)


df["goalkeeper_value_diff"] = df["goalkeeper_value_a"] - df["goalkeeper_value_b"]
df["defender_value_diff"] = df["defender_value_a"] - df["defender_value_b"]
df["midfield_value_diff"] = df["midfield_value_a"] - df["midfield_value_b"]
df["attack_value_diff"] = df["attack_value_a"] - df["attack_value_b"]


df["goalkeeper_share_diff"] = (
    safe_share(df["goalkeeper_value_a"], df["position_total_value_a"]) -
    safe_share(df["goalkeeper_value_b"], df["position_total_value_b"])
)

df["defender_share_diff"] = (
    safe_share(df["defender_value_a"], df["position_total_value_a"]) -
    safe_share(df["defender_value_b"], df["position_total_value_b"])
)

df["midfield_share_diff"] = (
    safe_share(df["midfield_value_a"], df["position_total_value_a"]) -
    safe_share(df["midfield_value_b"], df["position_total_value_b"])
)

df["attack_share_diff"] = (
    safe_share(df["attack_value_a"], df["position_total_value_a"]) -
    safe_share(df["attack_value_b"], df["position_total_value_b"])
)

df["goalkeeper_age_diff"] = safe_age_diff(df["goalkeeper_age_a"], df["goalkeeper_age_b"])
df["defender_age_diff"] = safe_age_diff(df["defender_age_a"], df["defender_age_b"])
df["midfield_age_diff"] = safe_age_diff(df["midfield_age_a"], df["midfield_age_b"])
df["attack_age_diff"] = safe_age_diff(df["attack_age_a"], df["attack_age_b"])

df["attack_defense_ratio_diff"] = (
    safe_ratio(df["attack_value_a"], df["defender_value_a"]) -
    safe_ratio(df["attack_value_b"], df["defender_value_b"])
)

df["midfield_attack_ratio_diff"] = (
    safe_ratio(df["midfield_value_a"], df["attack_value_a"]) -
    safe_ratio(df["midfield_value_b"], df["attack_value_b"])
)

df["total_age_a"] = mean_positive_age(
    df["goalkeeper_age_a"],
    df["defender_age_a"],
    df["midfield_age_a"],
    df["attack_age_a"],
)

df["total_age_b"] = mean_positive_age(
    df["goalkeeper_age_b"],
    df["defender_age_b"],
    df["midfield_age_b"],
    df["attack_age_b"],
)

df["total_age_diff"] = safe_age_diff(
    df["total_age_a"],
    df["total_age_b"],
)


print("Position value coverage:")
for col in [
    "goalkeeper_value_diff",
    "defender_value_diff",
    "midfield_value_diff",
    "attack_value_diff",
]:
    print(col, round((df[col] != 0).mean(), 3))

Position value coverage:
goalkeeper_value_diff 0.592
defender_value_diff 0.652
midfield_value_diff 0.649
attack_value_diff 0.659


## 7. Final Focused Feature Table

In [8]:
df["weighted_goals_for_diff_last5"] = (
    df["team_a_weighted_goals_for_avg_last5"] -
    df["team_b_weighted_goals_for_avg_last5"]
)

df["weighted_goals_against_diff_last5"] = (
    df["team_a_weighted_goals_against_avg_last5"] -
    df["team_b_weighted_goals_against_avg_last5"]
)

df["opponent_strength_diff_last5"] = (
    df["team_a_avg_opponent_elo_last5"] -
    df["team_b_avg_opponent_elo_last5"]
)

df["rating_change_diff_last5"] = (
    df["team_a_rating_change_avg_last5"] -
    df["team_b_rating_change_avg_last5"]
)

FEATURE_COLS = [
    # Elo / Ranking
    "rank_diff",
    "elo_diff",
    "rating_a_before",
    "rating_b_before",

    # Market Value
    "avg_player_value_diff",
    "log_market_value_a",
    "market_value_rel_mean_diff",

    # Recent Performance
    "opponent_strength_diff_last5",
    "weighted_goals_for_diff_last5",
    "weighted_goals_against_diff_last5",
    "rating_change_diff_last5",

    # Experience
    "team_a_matches_played_before",
    "team_b_matches_played_before",

    # Schedule
    "team_a_days_since_last_match",
    "team_b_days_since_last_match",

    # Tournament State
    "team_a_tournament_matches_played",
    "team_b_tournament_matches_played",
    "tournament_points_diff",
    "tournament_goal_diff_diff",

    # Position Features
    "goalkeeper_share_diff",
    "defender_share_diff",
]

TARGET_COLS = [
    "target_goals_a",
    "target_goals_b",
]

metadata_cols = [
    "date",
    "team_a",
    "team_b",
    "competition",
    "location",
    "season_id",
    "tournament_year",
    "tournament_key",
]

model_df = df[
    metadata_cols
    + FEATURE_COLS
    + TARGET_COLS
    + [
        "target_goal_diff",
        "target_total_goals",
    ]
].copy()

model_df["team_a_days_since_last_match"] = (
    model_df["team_a_days_since_last_match"]
    .clip(0, 60)
)

model_df["team_b_days_since_last_match"] = (
    model_df["team_b_days_since_last_match"]
    .clip(0, 60)
)

output_path = PROCESSED_DIR / "model_dataset.csv"

model_df.to_csv(
    output_path,
    index=False,
)

print("Final model dataset saved")
print("Shape:", model_df.shape)
print("Number of features:", len(FEATURE_COLS))
print("Saved to:", output_path)

display(model_df.head())

Final model dataset saved
Shape: (21540, 33)
Number of features: 21
Saved to: ../data/processed/model_dataset.csv


,date,team_a,team_b,competition,location,season_id,tournament_year,tournament_key,rank_diff,elo_diff,rating_a_before,rating_b_before,avg_player_value_diff,log_market_value_a,market_value_rel_mean_diff,opponent_strength_diff_last5,weighted_goals_for_diff_last5,weighted_goals_against_diff_last5,rating_change_diff_last5,team_a_matches_played_before,team_b_matches_played_before,team_a_days_since_last_match,team_b_days_since_last_match,team_a_tournament_matches_played,team_b_tournament_matches_played,tournament_points_diff,tournament_goal_diff_diff,goalkeeper_share_diff,defender_share_diff,target_goals_a,target_goals_b,target_goal_diff,target_total_goals
0,2004-01-01,Bermuda,Barbados,Dudley Eve Memorial Trophy,Bermuda,2004,2004,Dudley Eve Memorial Trophy_2004,52,-99,1270,1369,1.0,0.693147,0.016384,0.000000,0.0,0.000000,0.0,0,0,60,60,0,0,0,0,0.0,0.0,0,4,-4,4
1,2004-01-01,Kuwait,Yemen,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,-91,340,1519,1179,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0,0,60,60,0,0,0,0,0.0,0.0,4,0,4,4
2,2004-01-01,Saudi Arabia,Bahrain,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,-43,139,1662,1523,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0,0,60,60,0,0,0,0,0.0,0.0,1,0,1,1
3,2004-01-03,Bahrain,Oman,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,-10,-49,1511,1560,0.0,0.000000,0.000000,181.311955,0.0,0.890907,-12.0,1,0,2,60,1,0,0,-1,0.0,0.0,1,0,1,1
4,2004-01-03,Qatar,United Arab Emirates,Gulf Cup,Kuwait,2004,2004,Gulf Cup_2004,-8,66,1521,1455,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0,0,60,60,0,0,0,0,0.0,0.0,0,0,0,0


## 8. Dataset Validation

In [9]:
missing_values = model_df[FEATURE_COLS + TARGET_COLS].isna().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) > 0:
    display(missing_values)
    raise ValueError("Missing values found in model features or targets.")

non_numeric_features = [
    col for col in FEATURE_COLS
    if not pd.api.types.is_numeric_dtype(model_df[col])
]

if non_numeric_features:
    raise TypeError(f"Non-numeric feature columns found: {non_numeric_features}")

if (model_df[TARGET_COLS] < 0).any().any():
    raise ValueError("Negative goals found in target columns.")

print("Validation passed")
print("Rows:", len(model_df))
print("Features:", len(FEATURE_COLS))
print("Date range:", model_df["date"].min().date(), "to", model_df["date"].max().date())

Validation passed
Rows: 21540
Features: 21
Date range: 2004-01-01 to 2026-06-15


## 9. Poisson Score Conversion

In [10]:
def poisson_prob(lam, k):
    return (lam ** k) * exp(-lam) / factorial(k)

def most_likely_score(lambda_a, lambda_b, max_goals=6):
    lambda_a = min(max(float(lambda_a), 0.05), 4.0)
    lambda_b = min(max(float(lambda_b), 0.05), 4.0)

    best_score = None
    best_prob = -1

    for ga in range(max_goals + 1):
        for gb in range(max_goals + 1):
            prob = poisson_prob(lambda_a, ga) * poisson_prob(lambda_b, gb)

            if prob > best_prob:
                best_prob = prob
                best_score = (ga, gb)

    return best_score

## 10. Expanding-Window World Cup Backtest

In [11]:
model_df["is_world_cup"] = (
    model_df["competition"]
    .astype(str)
    .str.lower()
    .eq("world cup")
    .astype(int)
)

In [12]:
# =========================
# Final World Cup Backtest
# =========================

world_cup_years = sorted(
    model_df.loc[
        model_df["is_world_cup"] == 1,
        "date"
    ].dt.year.unique()
)

results = []
all_predictions = []

for year in world_cup_years:

    wc_df = model_df[
        (model_df["is_world_cup"] == 1) &
        (model_df["date"].dt.year == year)
    ].copy()

    wc_start = wc_df["date"].min()

    train_df = model_df[
        (model_df["date"] < wc_start) &
        (model_df["rating_a_before"] >= 1420) &
        (model_df["rating_b_before"] >= 1420)
    ].copy()

    test_df = wc_df.copy()

    X_train = train_df[FEATURE_COLS]
    y_train = train_df[TARGET_COLS]

    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_COLS]

    model = MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=600,
            max_depth=10,
            min_samples_leaf=8,
            random_state=42,
            n_jobs=-1
        )
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(y_test, preds)
    )

    mae = mean_absolute_error(
        y_test,
        preds
    )

    pred_scores = [
        most_likely_score(
            preds[i][0],
            preds[i][1]
        )
        for i in range(len(preds))
    ]

    actual_scores = [
        (
            int(y_test.iloc[i]["target_goals_a"]),
            int(y_test.iloc[i]["target_goals_b"])
        )
        for i in range(len(y_test))
    ]

    exact_hits = [
        pred_scores[i] == actual_scores[i]
        for i in range(len(actual_scores))
    ]

    direction_hits = []

    for pred, actual in zip(pred_scores, actual_scores):

        pred_result = (
            "A"
            if pred[0] > pred[1]
            else "B"
            if pred[0] < pred[1]
            else "D"
        )

        actual_result = (
            "A"
            if actual[0] > actual[1]
            else "B"
            if actual[0] < actual[1]
            else "D"
        )

        direction_hits.append(
            pred_result == actual_result
        )

    exact_acc = np.mean(exact_hits)
    direction_acc = np.mean(direction_hits)

    results.append({
        "world_cup_year": int(year),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "num_features": len(FEATURE_COLS),
        "rmse": rmse,
        "mae": mae,
        "exact_score_accuracy_%": round(exact_acc * 100, 2),
        "direction_accuracy_%": round(direction_acc * 100, 2),
    })

    pred_table = test_df[
        [
            "date",
            "team_a",
            "team_b",
            "target_goals_a",
            "target_goals_b",
        ]
    ].copy()

    pred_table["world_cup_year"] = int(year)
    pred_table["pred_goals_a"] = preds[:, 0]
    pred_table["pred_goals_b"] = preds[:, 1]

    pred_table["pred_score"] = [
        f"{a}-{b}"
        for a, b in pred_scores
    ]

    pred_table["actual_score"] = [
        f"{a}-{b}"
        for a, b in actual_scores
    ]

    pred_table["exact_hit"] = exact_hits

    all_predictions.append(pred_table)

results_df = pd.DataFrame(results)

predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

display(results_df)

print()
print("Average Exact Score Accuracy:",
      round(results_df["exact_score_accuracy_%"].mean(), 2),
      "%")

print("Average Direction Accuracy:",
      round(results_df["direction_accuracy_%"].mean(), 2),
      "%")

,world_cup_year,train_rows,test_rows,num_features,rmse,mae,exact_score_accuracy_%,direction_accuracy_%
0,2006,1217,64,21,0.940506,0.760407,15.62,82.81
1,2010,3122,64,21,0.964564,0.765198,20.31,85.94
2,2014,5088,64,21,1.062375,0.786572,7.81,79.69
3,2018,6894,64,21,0.925524,0.719126,20.31,85.94
4,2022,8691,64,21,0.970022,0.739333,29.69,81.25
5,2026,10084,1,21,0.389890,0.383633,0.00,100.00



Average Exact Score Accuracy: 15.62 %
Average Direction Accuracy: 85.94 %


## 11. Feature Importance

In [13]:
importances_a = model.estimators_[0].feature_importances_
importances_b = model.estimators_[1].feature_importances_

importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_goals_a": importances_a,
    "importance_goals_b": importances_b,
})

importance_df["avg_importance"] = (
    importance_df["importance_goals_a"]
    + importance_df["importance_goals_b"]
) / 2

importance_df = (
    importance_df
    .sort_values("avg_importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df)

,feature,importance_goals_a,importance_goals_b,avg_importance
0,rank_diff,0.483518,0.463559,0.473538
1,elo_diff,0.136976,0.130763,0.133870
2,rating_a_before,0.036677,0.104014,0.070345
3,rating_b_before,0.066620,0.028751,0.047686
4,avg_player_value_diff,0.030925,0.020125,0.025525
5,weighted_goals_for_diff_last5,0.022871,0.022622,0.022747
6,opponent_strength_diff_last5,0.022077,0.022954,0.022515
7,log_market_value_a,0.020526,0.023850,0.022188
8,weighted_goals_against_diff_last5,0.020741,0.020683,0.020712
9,team_a_matches_played_before,0.017890,0.021081,0.019486


## 11.1 Data Quality and Feature Validation


In [14]:
print("Dataset shape:", model_df.shape)
print("Date range:", model_df["date"].min(), "to", model_df["date"].max())

print("\nMissing values:")
missing = model_df.isna().sum()
display(missing[missing > 0].sort_values(ascending=False))

print("\nDuplicate rows:")
duplicates = model_df.duplicated(
    subset=["date", "team_a", "team_b", "competition"]
).sum()
print(duplicates)

print("\nTarget distribution:")
display(model_df[["target_goals_a", "target_goals_b", "target_goal_diff", "target_total_goals"]].describe())

print("\nFeature types:")
display(model_df[FEATURE_COLS].dtypes.value_counts())

non_numeric = [
    col for col in FEATURE_COLS
    if not pd.api.types.is_numeric_dtype(model_df[col])
]

print("Non numeric features:", non_numeric)

Dataset shape: (21540, 34)
Date range: 2004-01-01 00:00:00 to 2026-06-15 00:00:00

Missing values:


Series([], dtype: int64)


Duplicate rows:


3

Target distribution:


,target_goals_a,target_goals_b,target_goal_diff,target_total_goals
count,21540.000000,21540.000000,21540.000000,21540.000000
mean,1.790204,0.892572,0.897632,2.682776
std,1.653219,1.105457,2.101815,1.868878
min,0.000000,0.000000,-13.000000,0.000000
25%,1.000000,0.000000,0.000000,1.000000
50%,1.000000,1.000000,1.000000,2.000000
75%,2.000000,1.000000,2.000000,4.000000
max,21.000000,13.000000,21.000000,21.000000



Feature types:


int64      12
float64     9
Name: count, dtype: int64

Non numeric features: []


## 11.2 Transfermarkt Coverage Check

In [15]:
market_cols = [
    "log_market_value_a",
    "market_value_rel_mean_diff",
    "avg_player_value_diff",
]

coverage = {}

for col in market_cols:
    coverage[col] = (model_df[col] != 0).mean()

coverage_df = (
    pd.DataFrame.from_dict(
        coverage,
        orient="index",
        columns=["non_zero_share"]
    )
    .sort_values("non_zero_share", ascending=False)
)

display(coverage_df)

,non_zero_share
market_value_rel_mean_diff,0.943593
avg_player_value_diff,0.937372
log_market_value_a,0.887001


## 11.3 Correlation Check

In [16]:
corr = model_df[FEATURE_COLS].corr().abs()

high_corr_pairs = []

for i, col1 in enumerate(corr.columns):
    for col2 in corr.columns[i + 1:]:
        value = corr.loc[col1, col2]
        if value >= 0.90:
            high_corr_pairs.append({
                "feature_1": col1,
                "feature_2": col2,
                "abs_corr": value,
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values(
    "abs_corr", ascending=False
)

display(high_corr_df)

,feature_1,feature_2,abs_corr
0,rank_diff,elo_diff,0.945787
1,avg_player_value_diff,market_value_rel_mean_diff,0.914598


## 12. Save Evaluation Outputs

In [17]:
results_path = OUTPUTS_DIR / "world_cup_backtest_results.csv"
predictions_path = OUTPUTS_DIR / "world_cup_predictions.csv"
importance_path = OUTPUTS_DIR / "feature_importance.csv"

results_df.to_csv(results_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
importance_df.to_csv(importance_path, index=False)

print("Saved evaluation outputs:")
print("-", results_path)
print("-", predictions_path)
print("-", importance_path)

Saved evaluation outputs:
- ../outputs/evaluation/world_cup_backtest_results.csv
- ../outputs/evaluation/world_cup_predictions.csv
- ../outputs/evaluation/feature_importance.csv


## Summary

This notebook creates the final feature-engineering dataset for the World Cup score prediction model.

Main output:

- `data/processed/model_dataset.csv`

Evaluation outputs:

- `outputs/evaluation/world_cup_backtest_results.csv`
- `outputs/evaluation/world_cup_predictions.csv`
- `outputs/evaluation/feature_importance.csv`

The evaluation uses expanding-window World Cup backtesting:
each World Cup is predicted using only matches that occurred before that tournament.